In [2]:
#Ref
import pandas as pd
import glob
import numpy as np
import plotly.express as px
import re

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 120


# Pattern to match all relevant CSV files
file_pattern = path + "negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low*.csv"

# List all matching files
csv_files = glob.glob(file_pattern)

for f in csv_files:
    print(f)

# Extract H_low values
H_lows = []
for path in csv_files:
    match = re.search(r'H_low([0-9.]+)\.csv', path)
    if match:
        h_low = float(match.group(1))
        H_lows.append(h_low)

print(H_lows)

/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low2.8.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low2.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low2.5.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low0.5.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low3.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low1.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low1.5.csv
[2.8, 2.0, 2.5, 0.5, 3.0, 1.0, 1.5]


In [7]:
import plotly.graph_objects as go

n_cycles = 1200  # Adjust if needed
fig = go.Figure()

# Sort by numeric H_low extracted from the filename
csv_files = sorted(csv_files, key=lambda f: float(re.search(r'H_low([0-9.]+)\.csv', f).group(1)))

for file in csv_files:
    # Extract H_low from filename
    match = re.search(r'H_low([0-9.]+)\.csv', file)
    if not match:
        continue
    H_low_val = float(match.group(1))

    # Load CSV
    df_all = pd.read_csv(file)

    # Group and compute mean & SEM
    grouped = df_all.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()

    grouped['chi_sem'] = grouped['chi_std'] / (n_cycles ** 0.5)

    # Add trace for this H_low
    fig.add_trace(go.Scatter(
        x=grouped["i"],
        y=grouped["chi_mean"],
        error_y=dict(
            type="data",
            array=grouped["chi_sem"],
            visible=True
        ),
        mode="lines+markers",
        name=f"H_low = {H_low_val}" if H_low_val != 3.0 else "Ref (H_low = H_high)",
        hovertemplate=f"H_low={H_low_val}<br>i=%{{x}}<br>χ=%{{y:.3f}}<extra></extra>"
    ))

# Final layout
fig.update_layout(
    title="Chi vs MCS Step for Multiple H_low Values",
    xaxis_title="MCS Step (i)",
    yaxis_title="Mean χ",
    template="plotly_white",
)

fig.write_html('negative_field_cycle_H_high3.html')
fig.show()